In [ ]:
from math import exp, pi, sqrt

def gaussian(x, mean, std):
    return exp(-0.5 * ((x - mean) / std) ** 2) / (std * sqrt(2 * pi))



def map_setpoint(
    setpoint,
    price,
    prices_mean,
    prices_std,
    battery_energy,
    battery_min_limit,
    pv_power,
    house_power,
    max_feedin=4000,
    setpoint_spread=1,
    min_setpoint=-20,
):
    price = price * 100
    prices_mean = prices_mean * 100
    prices_std = prices_std * 100

    prices_std = max(5, prices_std)

    if price > prices_mean + prices_std:
        price = prices_mean + prices_std

    mean = prices_mean + prices_std
    std = setpoint_spread * prices_std

    max_prob = gaussian(0, 0, 2)
    gaus_prob = gaussian(price, mean, std) / max_prob * 1.5

    # print(f"p: {gaus_prob:.2f} price: {price:.2f} mean: {mean:.2f} std: {std:.2f} spread {setpoint_spread:.2f}")

    setpoint = gaus_prob * setpoint

    # quadratically going from 1 to 0 from battery_min_limit + 2 to battery_min_limit
    if battery_energy < battery_min_limit + 2:
        new_setpoint = setpoint * ((battery_energy - battery_min_limit) / 2) ** 4
        if pv_power > house_power:
            new_setpoint = max(setpoint, min(new_setpoint, -pv_power + house_power))
        setpoint = new_setpoint
    return max(-max_feedin, min(min_setpoint, setpoint))


: 